In [2]:
import folium
from folium import plugins
from shapely import wkt as shapely_wkt
import pandas as pd
import colorsys

# Load data
df = pd.read_csv("/Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/WORLDSAR/studies/WP2-Matching/S1_NISAR/nisar_gslc_20260203_143348.csv")

# Configuration
output_file = "nisar_gslc_footprints.html"
zoom_start = 4

def generate_colors(n):
    """Generate n distinct colors"""
    colors = []
    for i in range(n):
        hue = i / n
        rgb = colorsys.hsv_to_rgb(hue, 0.8, 0.9)
        hex_color = '#{:02x}{:02x}{:02x}'.format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))
        colors.append(hex_color)
    return colors

def plot_all_wkt(df, wkt_column='WKT', name_column='fileID', output_file='map.html', zoom_start=4):
    """
    Plot all WKT polygons from dataframe on a single map and save as HTML.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing WKT and name columns
    wkt_column : str
        Name of the column containing WKT strings
    name_column : str
        Name of the column containing polygon names
    output_file : str
        Output HTML filename
    zoom_start : int
        Initial zoom level for the map
    
    Returns:
    --------
    folium.Map
        Interactive folium map with all polygons
    """
    try:
        # Calculate overall centroid
        all_coords = []
        for idx, row in df.iterrows():
            try:
                geom = shapely_wkt.loads(row[wkt_column])
                centroid = geom.centroid
                all_coords.append((centroid.y, centroid.x))
            except:
                continue
        
        if not all_coords:
            print("No valid geometries found!")
            return None
            
        center_lat = sum(c[0] for c in all_coords) / len(all_coords)
        center_lon = sum(c[1] for c in all_coords) / len(all_coords)
        
        # Create base map
        m = folium.Map(
            location=[center_lat, center_lon],
            zoom_start=zoom_start,
            tiles='OpenStreetMap'
        )
        
        # Add additional tile layers
        folium.TileLayer('cartodbpositron', name='CartoDB Positron').add_to(m)
        folium.TileLayer('cartodbdark_matter', name='CartoDB Dark').add_to(m)
        
        # Generate colors for each polygon
        colors = generate_colors(len(df))
        
        # Track statistics
        successful = 0
        failed = 0
        
        # Add each polygon with its own feature group for layer control
        for idx, row in df.iterrows():
            try:
                # Parse WKT
                geom = shapely_wkt.loads(row[wkt_column])
                coords = [[lat, lon] for lon, lat in geom.exterior.coords]
                
                # Get name
                name = row[name_column] if name_column in df.columns else f"Polygon {idx}"
                color = colors[idx]
                
                # Calculate centroid
                centroid = geom.centroid
                center_lat_poly = centroid.y
                center_lon_poly = centroid.x
                
                # Create feature group for this polygon
                feature_group = folium.FeatureGroup(name=f"{idx}: {name[:30]}", show=True)
                
                # Create popup
                popup_html = f"""
                <div style="width:280px">
                    <h4>{name}</h4>
                    <b>Index:</b> {idx}<br>
                    <b>Centroid:</b> ({center_lat_poly:.4f}, {center_lon_poly:.4f})<br>
                </div>
                """
                
                # Add polygon to feature group
                folium.Polygon(
                    locations=coords,
                    color=color,
                    weight=2,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.3,
                    popup=folium.Popup(popup_html, max_width=300),
                    tooltip=f"{idx}: {name[:50]}"
                ).add_to(feature_group)
                
                # Add centroid marker to feature group
                folium.CircleMarker(
                    location=[center_lat_poly, center_lon_poly],
                    radius=3,
                    color=color,
                    fill=True,
                    fillColor=color,
                    fillOpacity=0.8,
                    popup=folium.Popup(f"<b>{name}</b><br>Index: {idx}", max_width=200),
                    tooltip=f"{idx}"
                ).add_to(feature_group)
                
                # Add feature group to map
                feature_group.add_to(m)
                
                successful += 1
                
            except Exception as e:
                print(f"Error plotting row {idx}: {e}")
                failed += 1
                continue
        
        # Add info box
        info_html = f'''
        <div style="position: fixed; 
                    top: 10px; left: 50px; width: 280px; 
                    background-color: white; border:2px solid grey; z-index:9999; 
                    font-size:12px; padding: 10px">
            <h4 style="margin-top:0">NISAR GSLC Footprints</h4>
            <b>Total Polygons:</b> {successful}<br>
            <b>Failed:</b> {failed}<br>
            <b>Center:</b> ({center_lat:.4f}, {center_lon:.4f})
        </div>
        '''
        m.get_root().html.add_child(folium.Element(info_html))
        
        # Add layer control
        folium.LayerControl().add_to(m)
        
        # Add fullscreen button
        plugins.Fullscreen().add_to(m)
        
        # Save to HTML
        m.save(output_file)
        print(f"Map saved to: {output_file}")
        print(f"Successfully plotted {successful} polygons")
        if failed > 0:
            print(f"Failed to plot {failed} polygons")
        
        return m
        
    except Exception as e:
        print(f"Error creating map: {e}")
        return None

# Plot all WKTs and save
m = plot_all_wkt(df, wkt_column='WKT', name_column='fileID', output_file=output_file, zoom_start=2)
if m:
    display(m)

Map saved to: nisar_gslc_footprints.html
Successfully plotted 6 polygons
